In [1]:
import pandas as pd

# Carga de datos de la Muestra 2 desde la carpeta data
df = pd.read_excel("../data/Muestras a Clavetear.xlsx", sheet_name="Muestra_2")

print(f"Dimensiones de la Muestra 2: {df.shape[0]} observaciones y {df.shape[1]} variables.")

Dimensiones de la Muestra 2: 2768 observaciones y 13 variables.


## 1. Análisis de Variable de Texto (`descripcion`)

In [2]:
desc_counts = df['descripcion'].value_counts()

print("--- ESTADÍSTICOS DE LA DESCRIPCIÓN ---")
print(f"Total de registros: {len(df)}")
print(f"Descripciones distintas (únicas): {len(desc_counts)}")
print(f"Descripciones que aparecen una sola vez: {(desc_counts == 1).sum()}")

print("\n--- TOP 5 DESCRIPCIONES MÁS FRECUENTES ---")
top_5 = desc_counts.head(5)
for desc, count in top_5.items():
    print(f"{count} veces | {desc}")

--- ESTADÍSTICOS DE LA DESCRIPCIÓN ---
Total de registros: 2768
Descripciones distintas (únicas): 772
Descripciones que aparecen una sola vez: 188

--- TOP 5 DESCRIPCIONES MÁS FRECUENTES ---
20 veces | PORSCHE 911 COUPE CARRERA 4 4X4 6VEL STD.
19 veces | PORSCHE 911 CONVERTIBLE CARRERA 4 4X4 6VEL STD.
15 veces | PORSCHE BOXTER CONVERTIBLE 6VEL STD.
14 veces | PORSCHE CAYENNE TURBO AUT.
14 veces | LINCOLN TOWN CAR "CARTIER" AUT.


**Interpretación de `descripcion`:**
La variable presenta una alta variabilidad. De las **772 descripciones únicas** en los 2,768 registros, **188 aparecen una sola vez** (valores poco frecuentes). No hay descripciones faltantes. 
Existe una concentración moderada en vehículos deportivos de lujo, siendo la descripción predominante *'PORSCHE 911 COUPE CARRERA 4 4X4 6VEL STD.'* (repetida 20 veces), lo cual sugiere que la flotilla de la Muestra 2 tiene un sesgo fuerte hacia gamas altas.

## 2. Análisis de Variables Cuantitativas (`modelo` y `cilindrada`)

In [3]:
cuantitativas = ['modelo', 'cilindrada']

print("--- ESTADÍSTICOS DESCRIPTIVOS ---")
# Calculamos estadísticos redondeando a 2 decimales
estadisticos = df[cuantitativas].describe().round(2)
print(estadisticos)

print("\n--- ANÁLISIS DE VALORES FALTANTES ---")
faltantes_cuant = df[cuantitativas].isna().sum()
porcentaje_falt_cuant = (faltantes_cuant / len(df)) * 100
print(pd.DataFrame({'Faltantes': faltantes_cuant, 'Porcentaje (%)': porcentaje_falt_cuant.round(2)}))

--- ESTADÍSTICOS DESCRIPTIVOS ---
        modelo  cilindrada
count  2768.00     1198.00
mean   2010.45        3.34
std       6.40        1.22
min    1983.00        1.40
25%    2006.00        2.50
50%    2012.00        3.00
75%    2016.00        3.70
max    2020.00        6.20

--- ANÁLISIS DE VALORES FALTANTES ---
            Faltantes  Porcentaje (%)
modelo              0            0.00
cilindrada       1570           56.72


**Interpretación de variables cuantitativas:**
- **`modelo`:** La flotilla abarca vehículos desde 1983 hasta 2020. No tiene valores faltantes. El 50% de los vehículos (rango intercuartílico entre el Q1 y Q3) se concentra entre los años-modelo 2006 y 2016, con un promedio de antigüedad situado en 2010.
- **`cilindrada`:** Oscila entre motores muy pequeños (1.4 L) y muy grandes (6.2 L), con una mediana de 3.0 L. **Problema importante de calidad de datos:** El 56.72% de los registros (1,570) no tiene este dato, por lo que los estadísticos solo representan a menos de la mitad de la muestra.

## 3. Análisis de Variables Cualitativas
Calculamos frecuencias absolutas y relativas (porcentajes) incluyendo los valores faltantes (`NaN`) dentro del total de la muestra para evaluar la integridad.

In [4]:
cualitativas = ['Marca', 'Tipo', 'Ocupantes', 'motor', 'transmision', 'traccion']

for col in cualitativas:
    print(f"\n{'='*40}")
    print(f"VARIABLE: {col}")
    print(f"{'='*40}")
    
    # Frecuencias absolutas y relativas incluyendo NaN
    abs_freq = df[col].value_counts(dropna=False)
    rel_freq = (df[col].value_counts(dropna=False, normalize=True) * 100).round(2)
    
    tabla_freq = pd.DataFrame({
        'Frecuencia Absoluta': abs_freq,
        'Porcentaje (%)': rel_freq
    })
    
    # Si hay muchas categorías, mostramos solo el Top 10 y agrupamos el resto
    if len(tabla_freq) > 10:
        top_10 = tabla_freq.head(10)
        resto_abs = tabla_freq['Frecuencia Absoluta'][10:].sum()
        resto_rel = tabla_freq['Porcentaje (%)'][10:].sum()
        
        resto_df = pd.DataFrame({
            'Frecuencia Absoluta': [resto_abs],
            'Porcentaje (%)': [resto_rel]
        }, index=['[OTROS TIPOS AGRUPADOS]'])
        
        print(pd.concat([top_10, resto_df]))
        print(f"\n*Nota: La variable tiene {df[col].nunique()} categorías distintas en total.")
    else:
        print(tabla_freq)


VARIABLE: Marca
                         Frecuencia Absoluta  Porcentaje (%)
PORSCHE                                  828           29.91
JEEP                                     655           23.66
CADILLAC                                 282           10.19
JAGUAR                                   226            8.16
SUBARU                                   212            7.66
LINCOLN                                  183            6.61
INFINITI                                 132            4.77
BUICK                                     76            2.75
ALFA ROMEO                                70            2.53
LAMBORGHINI                               44            1.59
[OTROS TIPOS AGRUPADOS]                   60            2.16

*Nota: La variable tiene 14 categorías distintas en total.

VARIABLE: Tipo
                         Frecuencia Absoluta  Porcentaje (%)
PORSCHE 911                              475           17.16
WRANGLER                                 278         

**Interpretación de cualitativas descriptivas y mecánicas:**
- **`Marca` y `Tipo`:** Existe una altísima concentración en vehículos **Porsche (29.91%)** y **Jeep (23.66%)**. El modelo predominante absoluto es el *PORSCHE 911* (17.16% de toda la base). Hay problemas de categorización en `Tipo`, donde el tercer valor más frecuente es la designación genérica *SEDAN* (8.06%).
- **`Ocupantes`:** Predominan los vehículos estándar de *5 OCUP.* (54.08%), seguidos de *4 OCUP.* (18.79%). No hay valores faltantes.
- **Variables mecánicas (problemas de integridad):** Hay una severa ausencia de datos en `traccion` (68.89% de nulos) y en `motor` (65.61% nulos). En contraste, `transmision` es una variable limpia, dominada ampliamente por la transmisión automática (*AUT* con 78.47%).

## 4. Análisis de Variables Cualitativas Dicotómicas (Binarias)

In [5]:
binarias = ['AC', 'Quemacocos', 'TURBO', 'hibrido']

for col in binarias:
    abs_freq = df[col].value_counts(dropna=False)
    rel_freq = (df[col].value_counts(dropna=False, normalize=True) * 100).round(2)
    
    tabla_freq = pd.DataFrame({
        'Frecuencia Absoluta': abs_freq,
        'Porcentaje (%)': rel_freq
    })
    print(f"\n--- {col} ---")
    print(tabla_freq)


--- AC ---
    Frecuencia Absoluta  Porcentaje (%)
AC                                     
0                  2086           75.36
1                   682           24.64

--- Quemacocos ---
            Frecuencia Absoluta  Porcentaje (%)
Quemacocos                                     
0                          2490           89.96
1                           278           10.04

--- TURBO ---
       Frecuencia Absoluta  Porcentaje (%)
TURBO                                     
0                     2337           84.43
1                      431           15.57

--- hibrido ---
         Frecuencia Absoluta  Porcentaje (%)
hibrido                                     
0                       2737           98.88
1                         31            1.12


**Interpretación de variables binarias:**
La integridad de las variables dicotómicas es perfecta (0% de valores faltantes). La gran mayoría de los vehículos de la muestra son de propulsión tradicional y aspiración natural:
- Solo el **1.12%** son vehículos híbridos.
- Solo el **10.04%** reporta tener quemacocos.
- El **15.57%** de la flota cuenta con motor turbo.
- Únicamente el **24.64%** tiene la bandera de Aire Acondicionado (`AC`) activada, lo que sugiere un posible subregistro considerando que la base está dominada por marcas premium.